# Ingestion des données — Hackathon Mobility
À attacher à ton Lakehouse (panneau de gauche > *Add lakehouse*) avant de lancer.
Télécharge les sources dans `Files/raw/`, puis crée les tables Delta : `arrets`, `gbfs_stations`, `carreaux_200m`, `communes`, `iris`.

In [ ]:
%pip install -q py7zr geopandas pyogrio

In [ ]:
import json, re, shutil, urllib.request
from datetime import datetime, timezone
from pathlib import Path

RAW = Path("/lakehouse/default/Files/raw")
SOURCES = {
    "arrets/gtfs-stops-france-export-2026-01-13.csv": "https://static.data.gouv.fr/resources/arrets-de-transport-en-france/20260113-104155/gtfs-stops-france-export-2026-01-13.csv",
    "population/Filosofi2021_carreaux_200m_gpkg.zip": "https://www.insee.fr/fr/statistiques/fichier/8735162/Filosofi2021_carreaux_200m_gpkg.zip",
    "projections/projdepreg_2021-2070_excel.zip": "https://www.insee.fr/fr/statistiques/fichier/6652134/projdepreg_2021-2070_excel.zip",
    "projections/projections_scenario_central.xlsx": "https://www.insee.fr/fr/statistiques/fichier/6652134/projections_scenario_central.xlsx",
    "gbfs/systems.csv": "https://raw.githubusercontent.com/MobilityData/gbfs/master/systems.csv",
    "admin/ADMIN-EXPRESS-COG_4-0__GPKG_WGS84G_FRA_2026-01-01.7z": "https://data.geopf.fr/telechargement/download/ADMIN-EXPRESS-COG/ADMIN-EXPRESS-COG_4-0__GPKG_WGS84G_FRA_2026-01-01/ADMIN-EXPRESS-COG_4-0__GPKG_WGS84G_FRA_2026-01-01.7z",
    "admin/CONTOURS-IRIS_3-0__GPKG_WGS84G_FRA_2026-01-01.7z": "https://data.geopf.fr/telechargement/download/CONTOURS-IRIS/CONTOURS-IRIS_3-0__GPKG_WGS84G_FRA_2026-01-01/CONTOURS-IRIS_3-0__GPKG_WGS84G_FRA_2026-01-01.7z",
}

def get(url):
    req = urllib.request.Request(url, headers={"User-Agent": "hackathon-mobility"})
    return urllib.request.urlopen(req, timeout=60)

for rel, url in SOURCES.items():
    dest = RAW / rel
    if dest.exists():
        continue
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + ".part")
    with get(url) as r, open(tmp, "wb") as f:
        shutil.copyfileobj(r, f)
    tmp.rename(dest)  # pas de fichier tronqué si le téléchargement casse
    print("ok", rel, dest.stat().st_size // 2**20, "Mo")

## Instantané GBFS
Chaque exécution écrit un dossier horodaté. Planifie ce notebook (ou cette cellule dans un notebook à part) toutes les 15-30 min pendant le hackathon :
l'historique de `station_status` donne un proxy d'usage par station, c'est la cible du modèle prédictif.

In [ ]:
KEEP = {"system_information", "station_information", "station_status", "free_bike_status", "vehicle_status", "vehicle_types"}
snap = RAW / "gbfs" / "snapshots" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%MZ")
index = []
for d in json.load(get("https://transport.data.gouv.fr/api/datasets")):
    res = next((r for r in d.get("resources", []) if r.get("format") == "gbfs"), None)
    if not res:
        continue
    slug = re.sub(r"[^a-z0-9]+", "-", d["title"].lower()).strip("-")[:80]
    try:
        root = json.load(get(res["url"]))
        data = root["data"]
        feeds = data["feeds"] if "feeds" in data else next(iter(data.values()))["feeds"]  # GBFS v3 / v2
        (snap / slug).mkdir(parents=True, exist_ok=True)
        saved = []
        for f in feeds:
            if f["name"] in KEEP:
                (snap / slug / f"{f['name']}.json").write_bytes(get(f["url"]).read())
                saved.append(f["name"])
        index.append({"dataset": d["title"], "slug": slug, "feeds": saved})
    except Exception as e:
        index.append({"dataset": d["title"], "slug": slug, "error": str(e)})
(snap / "index.json").write_text(json.dumps(index, ensure_ascii=False, indent=1))
print(sum("error" not in x for x in index), "flux OK /", len(index))

## Tables Delta

In [ ]:
# Arrêts GTFS (agrégat non dédupliqué : un même arrêt peut venir de plusieurs jeux)
arrets = (spark.read.option("header", True).option("multiLine", True).option("escape", '"')
          .csv("Files/raw/arrets/gtfs-stops-france-export-2026-01-13.csv"))
arrets.write.mode("overwrite").format("delta").saveAsTable("arrets")
print(arrets.count(), "arrêts")

In [ ]:
# Stations GBFS du dernier instantané (vélos, trottinettes, autopartage dont Citiz)
import pandas as pd

rows = []
for folder in sorted(p for p in snap.iterdir() if p.is_dir()):
    f = folder / "station_information.json"
    if not f.exists():
        continue
    for s in json.loads(f.read_text())["data"]["stations"]:
        name = s.get("name")
        if isinstance(name, list):  # GBFS v3 : noms localisés
            name = name[0]["text"] if name else None
        rows.append({"reseau": folder.name, "citiz": "citiz" in folder.name, "station_id": str(s["station_id"]),
                     "nom": name, "lat": s["lat"], "lon": s["lon"], "capacite": s.get("capacity")})
spark.createDataFrame(pd.DataFrame(rows)).write.mode("overwrite").format("delta").saveAsTable("gbfs_stations")
print(len(rows), "stations,", sum(r["citiz"] for r in rows), "Citiz")

In [ ]:
# Carroyage Filosofi 200 m -> centroïde WGS84 + géométrie WKT
import zipfile, geopandas as gpd

zip_path = RAW / "population/Filosofi2021_carreaux_200m_gpkg.zip"
parts = []
for name in zipfile.ZipFile(zip_path).namelist():
    if name.endswith(".gpkg"):
        g = gpd.read_file(f"zip://{zip_path}!{name}").to_crs(4326)
        c = g.geometry.centroid  # carreaux de 200 m : l'approximation en degrés est négligeable
        parts.append(pd.DataFrame(g.drop(columns="geometry")).assign(lat=c.y, lon=c.x, wkt=g.geometry.to_wkt(), zone=name))
carreaux = pd.concat(parts, ignore_index=True)
spark.createDataFrame(carreaux).write.mode("overwrite").format("delta").saveAsTable("carreaux_200m")
print(len(carreaux), "carreaux,", list(carreaux.columns))

In [ ]:
# Communes (ADMIN-EXPRESS) et IRIS
import py7zr, pyogrio

def load_layer(archive, pattern, table):
    out = archive.with_suffix("")
    if not out.exists():
        with py7zr.SevenZipFile(archive) as z:
            z.extractall(out)
    gpkg = next(out.rglob("*.gpkg"))
    layer = next(l for l, _ in pyogrio.list_layers(gpkg) if re.fullmatch(pattern, l, re.I))
    g = gpd.read_file(gpkg, layer=layer)
    df = pd.DataFrame(g.drop(columns="geometry")).assign(wkt=g.geometry.to_wkt())
    spark.createDataFrame(df).write.mode("overwrite").format("delta").saveAsTable(table)
    print(table, "<-", layer, len(df))

load_layer(RAW / "admin/ADMIN-EXPRESS-COG_4-0__GPKG_WGS84G_FRA_2026-01-01.7z", r"commune", "communes")
load_layer(RAW / "admin/CONTOURS-IRIS_3-0__GPKG_WGS84G_FRA_2026-01-01.7z", r".*iris.*", "iris")